## Cell 1 — Setup & Scientific Styling

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Output directory ──────────────────────────────────────────────────────────
OUT = Path('../results/scientific_plots')
OUT.mkdir(parents=True, exist_ok=True)

# ── Colorblind-friendly palette ───────────────────────────────────────────────
COLOR_ACTUAL  = "#2C3E50"
COLOR_XGB     = "#2980B9"
COLOR_CHRONOS = "#E74C3C"
COLOR_SARIMA  = "#F39C12"

# ── Global Matplotlib scientific style ───────────────────────────────────────
plt.rcParams.update({
    "figure.dpi":                    300,
    "savefig.dpi":                   300,
    "savefig.format":                "pdf",
    "font.family":                   "serif",
    "font.serif":                    ["Times New Roman", "DejaVu Serif", "Palatino"],
    "font.size":                     10,
    "axes.titlesize":                11,
    "axes.labelsize":                10,
    "xtick.labelsize":               9,
    "ytick.labelsize":               9,
    "legend.fontsize":               9,
    "legend.title_fontsize":         9,
    "axes.spines.top":               False,
    "axes.spines.right":             False,
    "axes.linewidth":                0.8,
    "axes.grid":                     True,
    "grid.color":                    "#CCCCCC",
    "grid.linewidth":                0.5,
    "grid.linestyle":                "--",
    "grid.alpha":                    0.6,
    "lines.linewidth":               1.4,
    "lines.antialiased":             True,
    "figure.constrained_layout.use": True,
    "mathtext.fontset":              "stix",
})

print("✓ Scientific style applied.")
print(f"✓ Output directory: {OUT.resolve()}")

## Cell 2 — Data Loading

In [ ]:
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd

# ── Path configuration ────────────────────────────────────────────────────────
ROOT_DIR = Path("..").resolve()

for p in [ROOT_DIR, ROOT_DIR / "src", ROOT_DIR / "src" / "features"]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from src.features.pipeline import build_features, get_feature_cols, TARGET_COLS
from src.features.scaling import split_and_scale

RAW_DIR       = Path("../data/raw").resolve()
PROCESSED_DIR = Path("../data/processed").resolve()
RESULTS_DIR   = Path("../results").resolve()

# ── 1. Build full feature DataFrame via src pipeline ─────────────────────────
try:
    # Support projects where master.parquet lands in processed/ not raw/
    if not (RAW_DIR / "master.parquet").exists() and (PROCESSED_DIR / "master.parquet").exists():
        print("ℹ master.parquet found in processed/. Directing pipeline to PROCESSED_DIR...")
        input_dir = PROCESSED_DIR
    else:
        input_dir = RAW_DIR
    df_full = build_features(input_dir)
    df_full.index = pd.to_datetime(df_full.index)
    print(f"✓ df_full built:  {df_full.shape}  [{df_full.index[0].date()} → {df_full.index[-1].date()}]")
except Exception as e:
    df_full = None
    print(f"✗ DATA NOT FOUND: could not build df_full  └─ {type(e).__name__}: {e}")

# ── 2. Derive df_test from the canonical train/val/test split ─────────────────
try:
    if df_full is None:
        raise RuntimeError("df_full is None — fix build_features() first.")
    feat_cols = get_feature_cols(df_full)
    splits = split_and_scale(
        df_full,
        target_cols=TARGET_COLS,
        feature_cols=feat_cols,
        train_end="2021-12-31",
        val_end="2022-12-31",
    )
    df_test = splits["test"]
    df_test.index = pd.to_datetime(df_test.index)
    print(f"✓ df_test split:  {df_test.shape}  [{df_test.index[0].date()} → {df_test.index[-1].date()}]")
except Exception as e:
    df_test = None
    print(f"✗ DATA NOT FOUND: could not derive df_test  └─ {type(e).__name__}: {e}")

# ── 3. Load evaluation metrics from ALL .jsonl files in results/ ──────────────
# Each notebook (XGBoost, Chronos, SARIMA …) saves its own .jsonl.
# Globbing the whole directory ensures no file is silently missed.
_records = []
try:
    if not RESULTS_DIR.exists():
        raise FileNotFoundError(f"Directory not found: {RESULTS_DIR}")
    _jsonl_files = sorted(RESULTS_DIR.glob("*.jsonl"))
    if not _jsonl_files:
        raise FileNotFoundError(f"No .jsonl files found in {RESULTS_DIR}")
    for metrics_file in _jsonl_files:
        _file_recs = [
            json.loads(line)
            for line in metrics_file.read_text().splitlines()
            if line.strip()
        ]
        print(f"  → {metrics_file.name}: {len(_file_recs)} record(s)")
        _records.extend(_file_recs)
    if not _records:
        raise ValueError(f"All .jsonl files in {RESULTS_DIR} were empty.")
    df_metrics = pd.DataFrame(_records)
    # Normalise: some notebooks log lowercase r2
    if "r2" in df_metrics.columns and "R2" not in df_metrics.columns:
        df_metrics = df_metrics.rename(columns={"r2": "R2"})
    # De-duplicate: keep last record when same model/target/split/horizon
    # appears in multiple files (e.g. after a re-run)
    _dedup_cols = [c for c in ["model", "target", "split", "horizon"] if c in df_metrics.columns]
    if _dedup_cols:
        _before = len(df_metrics)
        df_metrics = df_metrics.drop_duplicates(subset=_dedup_cols, keep="last")
        _dropped = _before - len(df_metrics)
        if _dropped:
            print(f"  ℹ Dropped {_dropped} duplicate row(s) (kept last occurrence).")
    print(f"✓ df_metrics loaded:  {df_metrics.shape}  | models: {sorted(df_metrics["model"].unique())}")
except Exception as e:
    df_metrics = None
    print(f"✗ DATA NOT FOUND: failed to load metrics  └─ {type(e).__name__}: {e}")

# ── 4. Load raw XGBoost prediction array (shape: [n_rows, 24]) ───────────────
_xgb_path = RESULTS_DIR / "xgb_preds_load_mw.npy"
try:
    if not _xgb_path.exists():
        raise FileNotFoundError(f"File not found: {_xgb_path}")
    xgb_preds = np.load(_xgb_path)
    print(f"✓ xgb_preds loaded:     {xgb_preds.shape}  dtype={xgb_preds.dtype}")
except Exception as e:
    xgb_preds = None
    print(f"✗ DATA NOT FOUND: {_xgb_path}  └─ {type(e).__name__}: {e}")

# ── 5. Load raw Chronos prediction array (shape: [n_rows, 24]) ───────────────
_chron_path = RESULTS_DIR / "chronos_preds_load_mw.npy"
try:
    if not _chron_path.exists():
        raise FileNotFoundError(f"File not found: {_chron_path}")
    chronos_preds = np.load(_chron_path)
    print(f"✓ chronos_preds loaded:  {chronos_preds.shape}  dtype={chronos_preds.dtype}")
except Exception as e:
    chronos_preds = None
    print(f"✗ DATA NOT FOUND: {_chron_path}  └─ {type(e).__name__}: {e}")

# ── Sanity-check: row alignment between df_test and prediction arrays ─────────
print()
_all_ok = all(v is not None for v in [df_full, df_test, df_metrics, xgb_preds, chronos_preds])
if _all_ok:
    _n      = len(df_test)
    _issues = []
    if xgb_preds.shape[0] != _n:
        _issues.append(f"xgb_preds rows ({xgb_preds.shape[0]}) ≠ df_test rows ({_n})")
    if chronos_preds.shape[0] != _n:
        _issues.append(f"chronos_preds rows ({chronos_preds.shape[0]}) ≠ df_test rows ({_n})")
    if _issues:
        print("⚠  Row-alignment warnings:")
        for msg in _issues:
            print(f"   └─ {msg}")
    else:
        print("✓ All datasets loaded and row-aligned — ready to plot.")
else:
    _missing = [
        name for name, val in {
            "df_full":       df_full,
            "df_test":       df_test,
            "df_metrics":    df_metrics,
            "xgb_preds":     xgb_preds,
            "chronos_preds": chronos_preds,
        }.items() if val is None
    ]
    print(f"⚠  {len(_missing)} dataset(s) unavailable: {chr(44).join(_missing)}")
    print("   Plots that depend on missing data will be skipped automatically.")

## Cell 3 — Plot 1: Macro Dataset Overview (Full Energy Mix)

In [ ]:
RENEWABLE_COLS = [
    "solar_mw",
    "wind_onshore_mw",
    "wind_offshore_mw",
    "biomass_mwh_smard",
    "run_of_river_mwh_smard",
    "pumped_storage_gen_mwh_smard",
    "other_renewables_mwh_smard",
]

RENEWABLE_LABELS = [
    "Solar PV",
    "Wind Onshore",
    "Wind Offshore",
    "Biomass",
    "Run-of-River Hydro",
    "Pumped Storage (Gen.)",
    "Other Renewables",
]

RENEWABLE_COLORS = [
    "#F9C846",
    "#5BA4CF",
    "#1B6CA8",
    "#6DBE72",
    "#3D9970",
    "#9B59B6",
    "#BDC3C7",
]

SPLIT_TRAIN_START = "2018-01-01"
SPLIT_VAL_START   = "2022-01-01"
SPLIT_TEST_START  = "2023-01-01"


def plot_macro_overview(df: pd.DataFrame, smooth_hours: int = 168) -> plt.Figure:
    """Stacked area chart of the full energy mix with train/val/test shading."""
    df_sm = df.rolling(smooth_hours, center=True, min_periods=1).mean()
    ys    = [df_sm[c].values for c in RENEWABLE_COLS]
    x     = df_sm.index
    load  = df_sm["load_mw"].values

    fig, ax = plt.subplots(figsize=(12, 5))

    shade_specs = [
        (SPLIT_TRAIN_START, SPLIT_VAL_START,  "#B0BEC5",    0.25, "Train (2018–2021)"),
        (SPLIT_VAL_START,   SPLIT_TEST_START, COLOR_SARIMA, 0.15, "Validation (2022)"),
        (SPLIT_TEST_START,  x[-1],            COLOR_CHRONOS, 0.12, "Test (2023–)"),
    ]
    shade_patches = []
    for t0, t1, color, alpha, label in shade_specs:
        ax.axvspan(pd.Timestamp(t0), pd.Timestamp(t1),
                   color=color, alpha=alpha, linewidth=0, zorder=0)
        shade_patches.append(
            mpatches.Patch(facecolor=color, alpha=max(alpha, 0.35),
                           edgecolor="none", label=label)
        )

    ax.stackplot(x, ys, labels=RENEWABLE_LABELS,
                 colors=RENEWABLE_COLORS, alpha=0.88, zorder=1)
    ax.plot(x, load, color=COLOR_ACTUAL, linewidth=1.6,
            linestyle="--", label="Grid Demand (Load)", zorder=3)

    ax.set_xlim(x[0], x[-1])
    ax.set_ylim(bottom=0)
    ax.set_xlabel("Date", labelpad=6)
    ax.set_ylabel("Power (MW / MWh)", labelpad=6)
    ax.set_title(
        f"German Energy Grid — Full Dataset Overview "
        f"({smooth_hours}-h Rolling Mean, 2018–2026)",
        pad=10,
    )
    ax.xaxis.set_major_locator(plt.matplotlib.dates.YearLocator())
    ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter("%Y"))
    plt.setp(ax.get_xticklabels(), rotation=0, ha="center")

    handles, labels_h = ax.get_legend_handles_labels()
    n_stack = len(RENEWABLE_COLS)
    leg1 = ax.legend(
        handles=handles[:n_stack], labels=labels_h[:n_stack],
        title="Renewable Generation", loc="upper left",
        framealpha=0.85, edgecolor="#AAAAAA", ncol=2,
        handlelength=1.2, columnspacing=0.8,
    )
    ax.add_artist(leg1)
    ax.legend(
        handles=[handles[-1]] + shade_patches,
        labels=[labels_h[-1]] + [p.get_label() for p in shade_patches],
        title="Series & Data Splits", loc="upper right",
        framealpha=0.85, edgecolor="#AAAAAA", handlelength=1.4,
    )
    return fig


if df_full is None:
    print("✗ Skipping Plot 1 — df_full not found. Load your data in Cell 2.")
else:
    _missing_cols = [c for c in RENEWABLE_COLS + ["load_mw"] if c not in df_full.columns]
    if _missing_cols:
        print(f"✗ Skipping Plot 1 — missing columns in df_full: {_missing_cols}")
    else:
        fig_macro = plot_macro_overview(df_full, smooth_hours=168)
        for ext in ("pdf", "png"):
            fig_macro.savefig(OUT / f"macro_overview.{ext}", bbox_inches="tight", dpi=300)
        plt.show()
        print("✓ Saved macro_overview.pdf / .png")

## Cell 4 — Plot 2: The 168-Hour Showdown

In [ ]:
def plot_showdown_168h(
    df_test:       pd.DataFrame,
    xgb_preds:     np.ndarray,
    chronos_preds: np.ndarray,
    horizon_step:  int = 1,
    n_hours:       int = 168,
) -> plt.Figure:
    """Compare Actual vs. XGBoost vs. Chronos for the first n_hours of the test set."""
    h_idx   = horizon_step - 1
    sl      = slice(0, n_hours)
    actual  = df_test["load_mw"].values[sl]
    xgb_fc  = xgb_preds[sl, h_idx]
    chron_fc= chronos_preds[sl, h_idx]
    time_ax = df_test.index[sl]

    fig, ax = plt.subplots(figsize=(12, 4))

    ax.plot(time_ax, actual,   color=COLOR_ACTUAL,  linewidth=1.8,
            label="Actual Load",                 zorder=3, alpha=0.95)
    ax.plot(time_ax, xgb_fc,   color=COLOR_XGB,     linewidth=1.4,
            linestyle="--",
            label=f"XGBoost (h={horizon_step})", zorder=2, alpha=0.90)
    ax.plot(time_ax, chron_fc, color=COLOR_CHRONOS, linewidth=1.4,
            label=f"Chronos (h={horizon_step})", zorder=4, alpha=0.90)

    for day_ts in pd.date_range(
        time_ax[0].normalize() + pd.Timedelta("1D"), time_ax[-1], freq="D"
    ):
        ax.axvline(day_ts, color="#DDDDDD", linewidth=0.6, zorder=0)

    ax.set_xlim(time_ax[0], time_ax[-1])
    ax.set_xlabel("Date / Hour (UTC)", labelpad=6)
    ax.set_ylabel("Load (MW)", labelpad=6)
    ax.set_title(
        f"168-Hour Forecast Showdown — XGBoost vs. Chronos "
        f"(Test Set, h={horizon_step})",
        pad=10,
    )
    ax.xaxis.set_major_locator(plt.matplotlib.dates.DayLocator())
    ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter("%b %d"))
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    ax.legend(
        loc="upper center", bbox_to_anchor=(0.5, -0.22),
        ncol=3, framealpha=0.9, edgecolor="#AAAAAA", handlelength=2.0,
    )
    return fig


_deps = {"df_test": df_test, "xgb_preds": xgb_preds, "chronos_preds": chronos_preds}
_missing = [k for k, v in _deps.items() if v is None]
if _missing:
    print(f"✗ Skipping Plot 2 — data not found: {', '.join(_missing)}")
else:
    fig_showdown = plot_showdown_168h(df_test, xgb_preds, chronos_preds,
                                      horizon_step=1, n_hours=168)
    for ext in ("pdf", "png"):
        fig_showdown.savefig(OUT / f"showdown_168h.{ext}", bbox_inches="tight", dpi=300)
    plt.show()
    print("✓ Saved showdown_168h.pdf / .png")

## Cell 5 — Plot 3: Horizon Error Propagation

In [ ]:
def compute_horizon_mae(actuals: np.ndarray, preds: np.ndarray) -> np.ndarray:
    """Per-horizon MAE. actuals shape (N,), preds shape (N, H)."""
    H   = preds.shape[1]
    mae = np.full(H, np.nan)
    for h in range(H):
        valid_n = len(actuals) - (h + 1)
        if valid_n <= 0:
            continue
        y_true  = actuals[h + 1 : h + 1 + valid_n]
        y_hat   = preds[:valid_n, h]
        mae[h]  = np.mean(np.abs(y_true - y_hat))
    return mae


def plot_horizon_error(
    df_test:       pd.DataFrame,
    xgb_preds:     np.ndarray,
    chronos_preds: np.ndarray,
) -> plt.Figure:
    """Line chart of MAE vs. forecast horizon (h = 1 … 24)."""
    actuals   = df_test["load_mw"].values
    xgb_mae   = compute_horizon_mae(actuals, xgb_preds)
    chron_mae = compute_horizon_mae(actuals, chronos_preds)
    horizons  = np.arange(1, xgb_preds.shape[1] + 1)

    fig, ax = plt.subplots(figsize=(8, 4))

    ax.plot(horizons, xgb_mae,   color=COLOR_XGB,     linewidth=1.8,
            marker="o", markersize=5, markeredgewidth=0.6,
            markeredgecolor="white", label="XGBoost (Recursive)")
    ax.plot(horizons, chron_mae, color=COLOR_CHRONOS, linewidth=1.8,
            marker="s", markersize=5, markeredgewidth=0.6,
            markeredgecolor="white", label="Chronos (Zero-Shot)")

    ax.set_xlim(0.5, len(horizons) + 0.5)
    ax.set_xticks(horizons)
    ax.set_xlabel("Forecast Horizon (hours ahead)", labelpad=6)
    ax.set_ylabel("MAE (MW)", labelpad=6)
    ax.set_title("Horizon Error Propagation — MAE vs. Forecast Step", pad=10)

    peak_h = int(np.nanargmax(xgb_mae))
    ax.annotate(
        "Recursive\nerror growth",
        xy=(horizons[peak_h], xgb_mae[peak_h]),
        xytext=(horizons[peak_h] - 4, xgb_mae[peak_h] + 0.04 * xgb_mae[peak_h]),
        arrowprops=dict(arrowstyle="->", color=COLOR_XGB, lw=1.1),
        fontsize=8, color=COLOR_XGB,
    )
    ax.legend(framealpha=0.9, edgecolor="#AAAAAA", loc="upper left")
    return fig


_deps = {"df_test": df_test, "xgb_preds": xgb_preds, "chronos_preds": chronos_preds}
_missing = [k for k, v in _deps.items() if v is None]
if _missing:
    print(f"✗ Skipping Plot 3 — data not found: {', '.join(_missing)}")
else:
    fig_horizon = plot_horizon_error(df_test, xgb_preds, chronos_preds)
    for ext in ("pdf", "png"):
        fig_horizon.savefig(OUT / f"horizon_error.{ext}", bbox_inches="tight", dpi=300)
    plt.show()
    print("✓ Saved horizon_error.pdf / .png")

## Cell 6 — Plot 4: Multi-Target R² Leaderboard

In [ ]:
TARGET_DISPLAY = {
    "load_mw":                      "Grid Load",
    "carbon_intensity_g_kwh":       "Carbon Intensity",
    "solar_mw":                     "Solar PV",
    "wind_onshore_mw":              "Wind Onshore",
    "wind_offshore_mw":             "Wind Offshore",
    "biomass_mwh_smard":            "Biomass",
    "run_of_river_mwh_smard":       "Run-of-River Hydro",
    "pumped_storage_gen_mwh_smard": "Pumped Storage",
    "other_renewables_mwh_smard":   "Other Renewables",
}

# ── Model detection ───────────────────────────────────────────────────────────
# Instead of hardcoding exact keys, we detect which model string in df_metrics
# contains "xgb" or "chronos" (case-insensitive) and map it to display config.
# This survives any pipeline naming variation: "xgboost", "xgb_recursive",
# "chronos-zs", "chronos-t5-small", "amazon-chronos", etc.

def _detect_model_keys(valid_models: set) -> dict:
    """Return {raw_key: {"label": str, "color": str}} for xgb + chronos."""
    result = {}
    for raw in sorted(valid_models):          # sorted for determinism
        low = raw.lower()
        if "xgb" in low or "xgboost" in low:
            if "xgb" not in {v.get("_id") for v in result.values()}:
                result[raw] = {"label": "XGBoost", "color": COLOR_XGB, "_id": "xgb"}
        elif "chronos" in low:
            if "chronos" not in {v.get("_id") for v in result.values()}:
                result[raw] = {"label": "Chronos", "color": COLOR_CHRONOS, "_id": "chronos"}
    return result


def plot_r2_leaderboard(df_metrics: pd.DataFrame) -> plt.Figure:
    """Horizontal grouped bar chart of R² across all targets for both models."""
    # Filter to test rows only when the split column exists
    if "split" in df_metrics.columns and "test" in df_metrics["split"].unique():
        df_plot = df_metrics[df_metrics["split"] == "test"].copy()
    else:
        df_plot = df_metrics.copy()

    valid_models = set(df_plot["model"].unique())
    model_cfg    = _detect_model_keys(valid_models)

    if not model_cfg:
        raise ValueError(
            f"No recognisable model keys found in df_metrics. "
            f"Raw values: {sorted(valid_models)}"
        )

    targets = list(TARGET_DISPLAY.keys())
    labels  = [TARGET_DISPLAY.get(t, t) for t in targets]
    n_t     = len(targets)
    n_m     = len(model_cfg)
    bar_h   = 0.35
    y_pos   = np.arange(n_t)

    fig, ax = plt.subplots(figsize=(9, 0.85 * n_t + 1.2))

    all_r2s = []

    for i, (raw_key, cfg) in enumerate(model_cfg.items()):
        df_m = df_plot[df_plot["model"] == raw_key].set_index("target")
        r2s  = [
            float(df_m.loc[t, "R2"]) if t in df_m.index else np.nan
            for t in targets
        ]
        all_r2s.extend([v for v in r2s if not np.isnan(v)])

        offset = (i - (n_m - 1) / 2) * bar_h
        bars   = ax.barh(
            y_pos + offset, r2s,
            height=bar_h * 0.88,
            color=cfg["color"], alpha=0.88,
            label=cfg["label"], zorder=2,
        )

        for bar, val in zip(bars, r2s):
            if np.isnan(val):
                continue
            if val >= 0:
                x_lbl, ha = bar.get_width() + 0.02, "left"
            else:
                x_lbl, ha = bar.get_width() - 0.02, "right"
            ax.text(
                x_lbl, bar.get_y() + bar.get_height() / 2,
                f"{val:.2f}", va="center", ha=ha,
                fontsize=7.5, color="#333333",
            )

    # Reference lines
    ax.axvline(0, color=COLOR_ACTUAL, linewidth=1.2, zorder=3)
    ax.axvline(1, color="#AAAAAA",    linewidth=0.8, linestyle=":", zorder=1)

    # Dynamic x-axis — left margin absorbs deeply negative bars + value labels
    min_x = min(all_r2s + [-0.15]) - 0.15
    ax.set_xlim(min_x, 1.12)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels)
    ax.invert_yaxis()
    ax.set_xlabel("$R^2$ Score (higher = better)", labelpad=6)
    ax.set_title("Multi-Target $R^2$ Leaderboard — XGBoost vs. Chronos", pad=10)
    ax.legend(loc="lower right", framealpha=0.9, edgecolor="#AAAAAA")

    for k in range(n_t):
        if k % 2 == 0:
            ax.axhspan(k - 0.5, k + 0.5, color="#F5F5F5", zorder=0)

    return fig


# ── Runner with full diagnostics ─────────────────────────────────────────────
if df_metrics is None:
    print("✗ Skipping Plot 4 — df_metrics not found. Load baseline_metrics.jsonl in Cell 2.")
else:
    _req_cols     = {"model", "target", "R2"}
    _missing_cols = _req_cols - set(df_metrics.columns)
    if _missing_cols:
        print(f"✗ Skipping Plot 4 — df_metrics is missing columns: {_missing_cols}")
    else:
        # ── Diagnostics: always print raw model keys before plotting ──────
        _raw_models    = sorted(df_metrics["model"].unique())
        _found_targets = sorted(df_metrics["target"].unique())
        print(f"─" * 52)
        print(f"df_metrics diagnostics")
        print(f"  Raw model keys : {_raw_models}")
        _detected = _detect_model_keys(set(_raw_models))
        print(f"  Detected as    : { {k: v['label'] for k, v in _detected.items()} }")
        _absent_targets = [t for t in TARGET_DISPLAY if t not in _found_targets]
        if _absent_targets:
            print(f"  ⚠  targets absent from metrics: {_absent_targets}")
        if len(_detected) < 2:
            print(f"  ⚠  Only {len(_detected)}/2 models detected — check raw model keys above.")
            print(f"     Expected substrings: 'xgb'/'xgboost' and 'chronos'.")
        else:
            print(f"  ✓ Both models detected.")
        print(f"  R2 range       : {df_metrics['R2'].min():.4f} → {df_metrics['R2'].max():.4f}")
        print(f"─" * 52)

        try:
            fig_r2 = plot_r2_leaderboard(df_metrics)
            for ext in ("pdf", "png"):
                fig_r2.savefig(OUT / f"r2_leaderboard.{ext}", bbox_inches="tight", dpi=300)
            plt.show()
            print("✓ Saved r2_leaderboard.pdf / .png")
            print(f"\nAll plots written to: {OUT.resolve()}")
        except ValueError as e:
            print(f"✗ Plot failed: {e}")

## Cell 7 — Plot 5: Carbon Intensity vs. Residual Load Density

In [ ]:
def plot_residual_load_dynamics(df: pd.DataFrame) -> plt.Figure:
    """Hexbin density plot of Residual Load vs Carbon Intensity with trendline."""
    ren_gen = df[[
        "solar_mw", "wind_onshore_mw", "wind_offshore_mw",
        "biomass_mwh_smard", "run_of_river_mwh_smard",
    ]].sum(axis=1)

    residual_load = df["load_mw"] - ren_gen
    carbon        = df["carbon_intensity_g_kwh"]

    valid = ~(residual_load.isna() | carbon.isna())
    x = residual_load[valid].values
    y = carbon[valid].values

    fig, ax = plt.subplots(figsize=(8, 5))
    hb = ax.hexbin(x, y, gridsize=40, cmap="YlOrRd",
                   mincnt=1, edgecolors="none", alpha=0.85, zorder=2)
    cb = fig.colorbar(hb, ax=ax, pad=0.02)
    cb.set_label("Density (Hours)", rotation=270, labelpad=15)

    # Linear trendline
    z = np.polyfit(x, y, 1)
    p = np.poly1d(z)
    x_trend = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_trend, p(x_trend), color=COLOR_ACTUAL,
            linestyle="--", linewidth=1.8, label="Linear Trend", zorder=3)

    ax.set_xlabel("Residual Load (MW) [Demand − Renewables]", labelpad=6)
    ax.set_ylabel("Carbon Intensity ($gCO_2eq / kWh$)", labelpad=6)
    ax.set_title(
        "Grid Physics — Carbon Intensity vs. Residual Load Dependency",
        pad=10,
    )
    ax.legend(loc="upper left", framealpha=0.9, edgecolor="#AAAAAA")
    return fig


_c7_required_cols = [
    "load_mw", "solar_mw", "wind_onshore_mw", "wind_offshore_mw",
    "biomass_mwh_smard", "run_of_river_mwh_smard", "carbon_intensity_g_kwh",
]
if df_full is None:
    print("✗ Skipping Plot 5 — df_full not loaded (see Cell 2).")
else:
    _c7_missing = [c for c in _c7_required_cols if c not in df_full.columns]
    if _c7_missing:
        print(f"✗ Skipping Plot 5 — missing columns in df_full: {_c7_missing}")
    else:
        fig_residual = plot_residual_load_dynamics(df_full)
        for ext in ("pdf", "png"):
            fig_residual.savefig(
                OUT / f"carbon_vs_residual_load.{ext}", bbox_inches="tight", dpi=300
            )
        plt.show()
        print("✓ Saved carbon_vs_residual_load.pdf / .png")

## Cell 8 — Plot 6: Seasonal & Diurnal Carbon Emissions Heatmap

In [ ]:
import seaborn as sns

def plot_carbon_shifting_heatmap(df: pd.DataFrame) -> plt.Figure:
    """Heatmap of mean carbon intensity by Month × Hour of Day."""
    df_c = df[["carbon_intensity_g_kwh"]].copy()
    df_c["month"] = df_c.index.month_name().str[:3]
    df_c["hour"]  = df_c.index.hour

    month_order = [
        "Jan", "Feb", "Mar", "Apr", "May", "Jun",
        "Jul", "Aug", "Sep", "Oct", "Nov", "Dec",
    ]
    pivot = df_c.pivot_table(
        values="carbon_intensity_g_kwh",
        index="month", columns="hour", aggfunc="mean",
    )
    pivot = pivot.reindex(month_order)

    fig, ax = plt.subplots(figsize=(10, 4.5))
    sns.heatmap(
        pivot,
        cmap="RdYlGn_r",
        annot=False,
        cbar_kws={"label": "Mean $gCO_2eq / kWh$"},
        linewidths=0.4,
        linecolor="#E0E0E0",
        ax=ax,
    )
    ax.set_xlabel("Hour of Day (UTC)", labelpad=6)
    ax.set_ylabel("Month", labelpad=6)
    ax.set_title(
        "Temporal Load Shifting Opportunity — Mean Carbon Intensity Profile",
        pad=10,
    )
    plt.setp(ax.get_yticklabels(), rotation=0)
    return fig


if df_full is None:
    print("✗ Skipping Plot 6 — df_full not loaded (see Cell 2).")
elif "carbon_intensity_g_kwh" not in df_full.columns:
    print("✗ Skipping Plot 6 — 'carbon_intensity_g_kwh' column missing from df_full.")
else:
    fig_heatmap = plot_carbon_shifting_heatmap(df_full)
    for ext in ("pdf", "png"):
        fig_heatmap.savefig(
            OUT / f"diurnal_seasonal_carbon_heatmap.{ext}", bbox_inches="tight", dpi=300
        )
    plt.show()
    print("✓ Saved diurnal_seasonal_carbon_heatmap.pdf / .png")

## Cell 9 — Plot 7: Chronos Probabilistic Uncertainty Intervals

In [ ]:
def plot_chronos_uncertainty(
    df_test:       pd.DataFrame,
    chronos_preds: np.ndarray,
    n_hours:       int = 48,
) -> plt.Figure:
    """Actual load vs Chronos h=1 forecast with shaded empirical 80% PI.

    The prediction interval is constructed from the empirical MAE over the
    plotted window (MAE × 1.28 ≈ one-sided 80% normal PI).  Replace with
    true quantile arrays if saved from the Chronos quantile pipeline.
    """
    sl        = slice(0, n_hours)
    actual    = df_test["load_mw"].values[sl]
    time_ax   = df_test.index[sl]
    median_fc = chronos_preds[sl, 0]          # h=1 point forecast

    empirical_mae = np.mean(np.abs(actual - median_fc))
    half_width    = empirical_mae * 1.28      # approx 80% symmetric interval
    upper_bound   = median_fc + half_width
    lower_bound   = median_fc - half_width

    fig, ax = plt.subplots(figsize=(10, 4))

    ax.fill_between(
        time_ax, lower_bound, upper_bound,
        color=COLOR_CHRONOS, alpha=0.15,
        label="~80% Prediction Interval (empirical)", zorder=1, lw=0,
    )
    ax.plot(time_ax, actual,    color=COLOR_ACTUAL,  linewidth=1.8,
            label="Actual Load",              zorder=3)
    ax.plot(time_ax, median_fc, color=COLOR_CHRONOS, linewidth=1.5,
            label="Chronos Median Forecast",  zorder=2)

    ax.set_xlim(time_ax[0], time_ax[-1])
    ax.set_xlabel("Time (UTC)", labelpad=6)
    ax.set_ylabel("Load (MW)", labelpad=6)
    ax.set_title(
        f"Foundation Model Uncertainty Quantification — {n_hours}-Hour Window",
        pad=10,
    )
    ax.xaxis.set_major_locator(plt.matplotlib.dates.HourLocator(interval=6))
    ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter("%b %d %H:%M"))
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    ax.legend(loc="upper right", framealpha=0.9, edgecolor="#AAAAAA")
    return fig


if df_test is None:
    print("✗ Skipping Plot 7 — df_test not loaded (see Cell 2).")
elif chronos_preds is None:
    print("✗ Skipping Plot 7 — chronos_preds not loaded (see Cell 2).")
elif chronos_preds.shape[1] < 1:
    print("✗ Skipping Plot 7 — chronos_preds has no horizon columns.")
else:
    fig_uncertainty = plot_chronos_uncertainty(df_test, chronos_preds, n_hours=48)
    for ext in ("pdf", "png"):
        fig_uncertainty.savefig(
            OUT / f"chronos_uncertainty_intervals.{ext}", bbox_inches="tight", dpi=300
        )
    plt.show()
    print("✓ Saved chronos_uncertainty_intervals.pdf / .png")

## Cell 10 — Plot 8: Feature Stream X-Ray (Missing Data Audit)
Visualizing sensor downtime, dropouts, and alignment gaps across the timeline before imputation.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

def plot_missing_data_heatmap(df: pd.DataFrame) -> plt.Figure:
    """Generates an X-ray heatmap of missing values across features."""
    fig, ax = plt.subplots(figsize=(10, 5))
    # Plot nullity matrix: True (missing) as bright contrast, False as dark baseline
    sns.heatmap(
        df.isnull(),
        cmap="cividis",
        cbar=False,
        yticklabels=int(len(df) / 10) if len(df) > 50 else True,
        ax=ax,
    )
    ax.set_title("Feature Stream X-Ray — Sensor Outages & Missing Data Audit", pad=15)
    ax.set_xlabel("Features / Sensors", labelpad=6)
    ax.set_ylabel("Timeline (Rows)", labelpad=6)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)

    # ── Legend: map cividis low/high ends to Present / Missing ───────────────
    # cividis: low values (False=0) → dark purple; high values (True=1) → bright yellow
    import matplotlib.cm as cm
    _cmap = cm.get_cmap("cividis")
    present_patch = mpatches.Patch(
        facecolor=_cmap(0.0), edgecolor="#555555", linewidth=0.5,
        label="Present (data available)"
    )
    missing_patch = mpatches.Patch(
        facecolor=_cmap(1.0), edgecolor="#555555", linewidth=0.5,
        label="Missing (NaN / dropout)"
    )
    ax.legend(
        handles=[present_patch, missing_patch],
        loc="upper right",
        framealpha=0.85,
        edgecolor="#AAAAAA",
        fontsize=8,
        title="Sensor Status",
        title_fontsize=8,
    )
    return fig

if df_full is None:
    print("✗ Skipping Plot 8 — df_full not found. Load your data in Cell 2.")
else:
    # Select a representative subset of base features to avoid plotting overlapping lag labels
    base_cols = [c for c in df_full.columns if not any(sub in c for sub in ["lag", "rolling", "diff"])]
    df_audit = df_full[base_cols] if base_cols else df_full

    fig_missing = plot_missing_data_heatmap(df_audit)
    for ext in ("pdf", "png"):
        fig_missing.savefig(OUT / f"missing_data_heatmap.{ext}", bbox_inches="tight", dpi=300)
    plt.show()
    print("✓ Saved missing_data_heatmap.pdf / .png")
